# Space-map — Xenium (polyp) with cell types (multi-channel)

One-click **multi-channel** registration: the density (all cells) drives
feature matching, while each of the top-N cell types forms an auxiliary
channel that **weights the choice of the best affine matrix** (weight =
that type's cell count). All channels are stored on each slice and
transform together; the non-rigid step then uses density only.

**Data:** set `DATA` to your local `xenium_polyp.csv.gz`
(columns `x`, `y`, `layer`, `cell_type`). Run all cells.

In [ ]:
import os, sys
# Ensure this repo's space_map wins over any editable/site-packages install.
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import space_map
from space_map import Slice
from space_map.flow import FlowImport, FlowExport, AutoFlowMultiCenter4

# --- edit this to your local file ---
DATA = 'data/xenium_polyp.csv.gz'
LAYER_COL, CELLTYPE_COL = 'layer', 'cell_type'
X_COL, Y_COL = 'x', 'y'
WORKDIR = 'out/xenium_celltype/_work'
TOP_N = 10  # most abundant cell types kept as auxiliary channels per slice

## Multi-channel import

`FlowImport.init_from_codex_multi` groups rows by layer, keeps the
`cell_type` column, and expands it to one-hot `ct::<type>` channels
(top-N by count). Coordinate normalization is computed from density and
applied identically to every channel.

In [ ]:
fi = FlowImport(WORKDIR)
slices = fi.init_from_codex_multi(
    DATA, layer_col=LAYER_COL, celltype_col=CELLTYPE_COL,
    x_col=X_COL, y_col=Y_COL, top_n=TOP_N)
print(f'{len(slices)} slices imported')
s0 = slices[0]
print('channels on slice0:', sorted(s0.imgs.keys()))
print('top cell types (label, count):')
for lab, c in s0.celltype_topk():
    print(f'   {c:>8d}  {lab}')

## Affine — density features, cell-type-weighted selection

In [ ]:
af = AutoFlowMultiCenter4(slices, initJKey=Slice.rawKey, alignMethod='auto')
af.affine(useKey='DF', show=False)
print('affine done')

## Non-rigid (density only) and export all channels

In [ ]:
af.ldm_pair(Slice.align1Key, Slice.align2Key, show=False)
KEY = Slice.align2Key  # final (non-rigid) alignment; use Slice.align1Key for affine-only

fe = FlowExport(slices)

# 1) aligned points for every channel (density + each cell type) as a long table
table = fe.export_all_channels(KEY, path='out/xenium_celltype/all_channels.csv')
print('exported', table.shape, 'rows'); print(table.groupby('channel').size())

# 2) Imaris: per-layer, per-channel single-channel TIFFs (1000x1000)
#    -> out/xenium_celltype/imaris/  named c_<layer>_<channel>.tiff
fe.export_imaris_tiff('out/xenium_celltype/imaris', key=KEY, size=1000)

# 3) 3D reconstruction colored by cell type (~45 deg, layers spaced along Z)
fe.plot_3d(key=KEY, save='out/xenium_celltype/reconstruction_3d_celltype.png',
           title='Xenium polyp - 3D by cell type')
print('wrote imaris/ and reconstruction_3d_celltype.png')

## Visualize: density vs. one cell-type channel, aligned

In [ ]:
import matplotlib.pyplot as plt
ct_keys = [k for k in s0.imgs if k.startswith('ct::')]
ct = ct_keys[0] if ct_keys else 'DF'
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for s in slices:
    p = s.imgs['DF'].get_points(Slice.align2Key)
    ax[0].scatter(p[:, 0], p[:, 1], s=0.2, alpha=0.3)
ax[0].set_title('density (aligned)'); ax[0].set_aspect('equal')
for s in slices:
    p = s.imgs[ct].get_points(Slice.align2Key)
    ax[1].scatter(p[:, 0], p[:, 1], s=0.3, alpha=0.4)
ax[1].set_title(f'{ct} (aligned)'); ax[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## Inspect the cell-type 3D reconstruction inline

In [ ]:
from IPython.display import Image
Image(filename='out/xenium_celltype/reconstruction_3d_celltype.png')

## Command-line equivalent

The same multi-channel flow can be run without Jupyter (from the
`examples/` directory):

```bash
python run_all.py celltype --data data/xenium_polyp.csv.gz -o out --top-n 10
```